In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [4]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [5]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [6]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 정밀 보정 (Raw Data Inspection)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 분석 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [진단] 412번 인덱스의 '원본(Raw)' 좌표 확인
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> [진단] Raw Index 412: {raw_412}")
                    # 예상: [418.xx, 0.0, -308.xx] 또는 [418.xx, -308.xx, 0.0] 등
                    
                    # 3) [해결] 목표 좌표(Target)와 비교하여 매핑 결정
                    # Target Z: -308.20
                    
                    # Case A: Raw Y가 -308 근처인 경우 -> Y를 Z로 (x, 0, y)
                    if np.isclose(raw_412[1], -308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 Z축으로 이동합니다. (Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]
                        
                    # Case B: Raw Z가 -308 근처인 경우 -> 그대로 사용 (Z -> Z)
                    elif np.isclose(raw_412[2], -308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 2]
                        
                    # Case C: Raw Y가 308 근처인 경우 -> 부호 반전 후 이동 (-Y -> Z)
                    elif np.isclose(raw_412[1], 308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 반전하여 Z축으로 이동합니다. (-Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 1]

                    # Case D: Raw Z가 308 근처인 경우 -> 부호 반전 (-Z -> Z)
                    elif np.isclose(raw_412[2], 308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 반전하여 사용합니다. (-Z -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 2]
                        
                    else:
                        print(" -> [경고] 자동 매핑 실패. Raw 데이터가 예상과 다릅니다. 원본 그대로 사용합니다.")
                        # 기본: (x, 0, y) 시도 (가장 흔한 패턴)
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]

                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
            break

[보정] 빨간색 도로 좌표 정밀 분석 시작...
 -> [진단] Raw Index 412: [ 4.1827621e+02  1.8871996e-14 -3.0820306e+02]
 -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)
 -> 좌표 변환 완료 (610 vertices)


bs 1
ue 7
V 

In [7]:
# ==========================================
# 1. 파일 저장 경로 설정 (요청 사항 반영)
# ==========================================
# 저장할 디렉토리 경로
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"

# 디렉토리가 없으면 생성
if not os.path.exists(output_dir):
    try:
        os.makedirs(output_dir)
        print(f"디렉토리 생성됨: {output_dir}")
    except OSError as e:
        print(f"[오류] 디렉토리를 생성할 수 없습니다: {e}")
        # 실패 시 현재 디렉토리에 저장하도록 fallback
        output_dir = "."

# Scene 로딩을 위한 경로 설정 (기존 유지)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

In [8]:
import os
import tensorflow as tf
import numpy as np
import sionna
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# [버전 호환성] Import 경로 처리
try:
    from sionna.phy.ofdm import ResourceGrid
except ImportError:
    from sionna.ofdm import ResourceGrid



# ==========================================
# 1. 파일 저장 경로 및 Scene 설정
# ==========================================
# 저장할 디렉토리 경로
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"
os.makedirs(output_dir, exist_ok=True)

xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
temp_xml_path = os.path.join(os.path.dirname(xml_path), "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 및 시뮬레이션 파라미터
# ==========================================
carrier_frequency = 3.5e9
subcarrier_spacing = 30e3
fft_size = 72
dt = 0.5e-3
total_steps = 20000  # 10초(0.5ms step)면 20000 step

# ✅ BS 위치 변경 (요청 반영)
tx_positions = [
    [-125.663, 100.367, -101.453],
    [-50.472,  100.869, -181.315],
    [0.663,    100.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

# UE 속도 및 개수
speeds_kmh = [30, 40, 50, 60, 70, 80, 90, 100]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

# ==========================================
# 3. 객체 및 경로 정의 (PolylineWalker)
# ==========================================
# road_positions는 이전 세션에서 로드된 상태여야 합니다.

# ✅ 경로 인덱스 변경 (요청 반영)
path_indices = [
    514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284,
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120,
    117, 116, 525, 582
]

trajectory_points = road_positions[path_indices]

class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)

    def get_position(self, distance):
        total_len = self.cum_dist[-1]
        if distance >= total_len:
            return self.points[-1]
        if distance <= 0:
            return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return self.points[idx] + (self.points[idx+1] - self.points[idx]) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성
# ==========================================
scene = load_scene(temp_xml_path)
scene.tx_array = PlanarArray(
    num_rows=8, num_cols=8,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V"
)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    pattern="iso", polarization="V"
)

# BS 추가
for name, pos in zip(tx_names, tx_positions):
    scene.add(Transmitter(name=name, position=pos))

# UE 추가 (초기 위치 = trajectory 시작점)
ues = []
for i in range(num_ues):
    rx = Receiver(name=f"UE_{i}", position=trajectory_points[0])
    scene.add(rx)
    ues.append(rx)

# ==========================================
# 5. 시뮬레이션 루프
# ==========================================
solver = PathSolver()
dataset_h = []

print("시뮬레이션 시작...")

for step in range(total_steps):
    current_time = step * dt

    # UE 이동 업데이트
    for i, ue in enumerate(ues):
        ue.position = walker.get_position(speeds_ms[i] * current_time)

    # Ray Tracing
    paths = solver(scene, max_depth=3)
    frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)

    cfr_output = paths.cfr(frequencies=frequencies)

    # [1] 튜플/단일 텐서 여부 판별 후 Numpy 변환
    if isinstance(cfr_output, (list, tuple)):
        h_real = cfr_output[0].numpy()
        h_imag = cfr_output[1].numpy()
    else:
        h_np = cfr_output.numpy()
        h_real = h_np.real
        h_imag = h_np.imag

    # [2] 마지막 축에 Real/Imag 쌓기
    h_stacked = np.stack([h_real, h_imag], axis=-1)

    # [3] 크기 1인 차원 자동 제거
    h_squeezed = np.squeeze(h_stacked)

    dataset_h.append(h_squeezed)

    # 디버깅 출력
    if step == 0:
        print(f"Original Stacked Shape: {h_stacked.shape}")
        print(f"Squeezed Shape: {h_squeezed.shape}")

    if step % 1000 == 0:
        print(f"Step {step}/{total_steps} 완료")

# ==========================================
# 6. 저장
# ==========================================
dataset_h = np.array(dataset_h)
save_path = os.path.join(output_dir, "nlos_bs3_ue8_channel_dataset_precise_10s.npy")
np.save(save_path, dataset_h)

print("-" * 30)
print(f"저장 완료. 최종 데이터 형태: {dataset_h.shape}")
print(f"저장 경로: {save_path}")


2026-02-10 16:18:30.337156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770707910.352473 1884292 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770707910.357169 1884292 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770707910.369995 1884292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770707910.370006 1884292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770707910.370009 1884292 computation_placer.cc:177] computation placer alr

시뮬레이션 시작...


jit_flush_malloc_cache(): Dr.Jit exhausted the available memory and had to flush its allocation cache to free up additional memory. This is an expensive operation and will have a negative effect on performance. You may want to change your computation so that it uses less memory. This warning will only be displayed once.


RuntimeError: dr.while_loop(): encountered an exception (see above).